In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_E.xpt"
csv_file = DATA_DIR / "HUQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 HUQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ040",
    "HUQ050",
    "HUQ060",
    "HUQ071",
    "HUD080",
    "HUQ090"
]


if df.shape != (10149, 10):
    raise ValueError(
        f"Unexpected HUQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "HUQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: HUQ_E structure = "
    "10,149 rows x 10 columns."
)


# ============================================================
# 4. CDC CHECK ITEMS
# ============================================================

#
# CDC HTML includes:
#
# HUQ015 - CHECK ITEM
# HUQ085 - CHECK ITEM
#
# They are questionnaire-routing instructions,
# not released variables in HUQ_E.xpt.
#
# DO NOT add them to CSV.
#

CHECK_ITEMS = [
    "HUQ015",
    "HUQ085"
]


for col in CHECK_ITEMS:

    if col in df.columns:
        raise ValueError(
            f"Unexpected CHECK ITEM in XPT: {col}"
        )


print(
    "PASS: HUQ015 and HUQ085 correctly absent "
    "from released XPT."
)


# ============================================================
# 5. FIELD CLASSIFICATION
# ============================================================

#
# Actual XPT inspection:
#
# Genuine decimal variables = 0
#
# All released variables contain whole-number:
#
# - identifiers
# - questionnaire codes
# - counts/categories
#
# Therefore all can use Int64 after the validated
# HUQ050 tiny -> zero correction.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()

DECIMAL_COLUMNS = []


print(
    "\nInteger/code/count fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 6. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATE EXACT TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "HUQ050": 1449
}

EXPECTED_TINY_TOTAL = 1449


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected true numeric zeros already "
        "decoded in original XPT."
    )


print(
    "\nPASS: Exactly 1,449 tiny artifacts "
    "found, all in HUQ050."
)


# ============================================================
# 8. CDC ZERO-SAFETY VALIDATION
# ============================================================

#
# CDC HUQ050:
#
# 0  None          = 1,449
# 1  1 visit       = 1,957
# 2  2-3 visits    = 2,946
# 3  4-9 visits    = 2,638
# 4  10-12 visits  =   604
# 5  13+ visits    =   546
# 77 Refused       =     1
# 99 Don't know    =     8
#
# Therefore the exact HUQ050 tiny values represent
# source zeros and may safely be restored to 0.
#

if set(actual_tiny_counts) != {"HUQ050"}:

    raise ValueError(
        "STOP: Tiny values found outside HUQ050."
    )


print(
    "PASS: CDC confirms HUQ050 permits "
    "0 = None."
)


# ============================================================
# 9. RESTORE HUQ050 ZEROS
# ============================================================

mask = (
    df["HUQ050"]
    .eq(TINY_VALUE)
)


if int(
    mask.sum()
) != 1449:

    raise ValueError(
        "HUQ050 tiny count differs."
    )


df.loc[
    mask,
    "HUQ050"
] = 0.0


if int(
    df["HUQ050"]
    .eq(0)
    .sum()
) != 1449:

    raise ValueError(
        "HUQ050 restored-zero count differs."
    )


print(
    "\nPASS: HUQ050 1,449 tiny values -> 0."
)


# ============================================================
# 10. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 12. FRACTIONAL / DECIMAL AUDIT
# ============================================================

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 13. CDC FREQUENCY EXPECTATIONS
# ============================================================

EXPECTED_FREQUENCIES = {

    "HUQ010": {
        1: 2790,
        2: 2513,
        3: 3032,
        4: 1445,
        5: 366,
        7: 1,
        9: 2
    },

    "HUQ020": {
        1: 1669,
        2: 836,
        3: 7159,
        9: 2
    },

    "HUQ030": {
        1: 8933,
        2: 1183,
        3: 32,
        9: 1
    },

    "HUQ040": {
        1: 1848,
        2: 6607,
        3: 258,
        4: 161,
        5: 90,
        9: 1
    },

    "HUQ050": {
        0: 1449,
        1: 1957,
        2: 2946,
        3: 2638,
        4: 604,
        5: 546,
        77: 1,
        99: 8
    },

    "HUQ060": {
        1: 91,
        2: 80,
        3: 812,
        4: 418,
        5: 47,
        9: 1
    },

    "HUQ071": {
        1: 1040,
        2: 9107,
        9: 2
    },

    "HUQ090": {
        1: 638,
        2: 8211
    }
}


EXPECTED_MISSING = {

    "HUQ010": 0,
    "HUQ020": 483,
    "HUQ030": 0,
    "HUQ040": 1184,
    "HUQ050": 0,
    "HUQ060": 8700,
    "HUQ071": 0,
    "HUQ090": 1300
}


# ============================================================
# 14. CDC FREQUENCY CHECK
# ============================================================

print(
    "\n--- CDC FREQUENCY CHECK ---"
)


for col, expected_counts in (
    EXPECTED_FREQUENCIES.items()
):

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"{col} frequency mismatch.\n"
            f"Expected: {expected_counts}\n"
            f"Actual:   {actual_counts}"
        )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_missing != EXPECTED_MISSING[col]:

        raise ValueError(
            f"{col} missing count differs.\n"
            f"Expected: {EXPECTED_MISSING[col]:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 15. SPECIAL CHECK - HUD080
# ============================================================

#
# Number of overnight hospital admissions.
#
# CDC:
#
# 1-5   = 1,025
# 6     = 13  (6 or more)
# 77777 = 0
# 99999 = 2
# Missing = 9,109
#
# Important:
#
# HUD080 is a derived/public-release field with
# an upper category of 6 or more.
#
# DO NOT attempt to reconstruct counts above 6.
#

s = df["HUD080"]


if int(
    s.between(
        1,
        5
    ).sum()
) != 1025:

    raise ValueError(
        "HUD080 1-5 count differs."
    )


if int(
    s.eq(6)
    .sum()
) != 13:

    raise ValueError(
        "HUD080=6 count differs."
    )


if int(
    s.eq(77777)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected HUD080=77777."
    )


if int(
    s.eq(99999)
    .sum()
) != 2:

    raise ValueError(
        "HUD080=99999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 9109:

    raise ValueError(
        "HUD080 missing count differs."
    )


print(
    "\nPASS: HUD080 matches CDC."
)

print(
    "      1-5 = 1,025"
)

print(
    "      6 or more = 13"
)

print(
    "      99999 = 2"
)

print(
    "      Missing = 9,109"
)


# ============================================================
# 16. IMPORTANT CDC RELEASE NOTE
# ============================================================

#
# CDC notes:
#
# HUD080 is released with an upper limit of
# "6 or more".
#
# The 2007-2008 interviewer instruction clarified that
# the count means hospital admissions, NOT total nights.
#
# Preserve released XPT values exactly.
#

print(
    "\nPASS: HUD080 public-release coding "
    "will remain unchanged."
)


# ============================================================
# 17. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY:
#
# HUQ050 tiny -> 0
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["HUQ050"]
    .eq(TINY_VALUE),
    "HUQ050"
] = 0.0


# ============================================================
# 18. VERIFY ALL RELEASED FIELDS ARE WHOLE
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):

        raise ValueError(
            f"STOP: {col} contains a non-whole value."
        )


print(
    "PASS: All 10 released HUQ_E variables "
    "contain whole-number values after correction."
)


# ============================================================
# 19. CLEAN INTEGER FORMATTING
# ============================================================

#
# Examples:
#
# SEQN
# 41475.0 -> 41475
#
# HUQ050
# tiny -> 0
#
# HUD080
# 6.0 -> 6
# 99999.0 -> 99999
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All HUQ_E variables converted "
    "to clean integer-style formatting."
)


# ============================================================
# 20. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected pre-export changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY HUQ050 tiny -> zero."
)


# ============================================================
# 21. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
# round()
# float_format
# arbitrary recoding
# missing-value filling
#
# Missing XPT values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 22. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 23. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 24. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 10 columns numerically match "
    "expected XPT values exactly."
)


# ============================================================
# 25. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 26. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 10149:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 27. STRICT RAW CSV TOKEN FIDELITY
# ============================================================

#
# Every nonmissing value should use:
#
# str(int(source_value))
#
# after ONLY HUQ050 tiny -> 0.
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:
        print(example)


    raise ValueError(
        "HUQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 28. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 29. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 30. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 31. FINAL HUQ050 ZERO CHECK
# ============================================================

final_huq050_zeros = int(
    csv_df["HUQ050"]
    .eq(0)
    .sum()
)


if final_huq050_zeros != 1449:

    raise ValueError(
        "Final HUQ050 zero count differs."
    )


print(
    "PASS: Final HUQ050 zero count = 1,449."
)


# ============================================================
# 32. FINAL CDC FREQUENCY RECHECK
# ============================================================

for col, expected_counts in (
    EXPECTED_FREQUENCIES.items()
):

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"Final {col} frequencies changed."
        )


    if int(
        csv_df[col]
        .isna()
        .sum()
    ) != EXPECTED_MISSING[col]:

        raise ValueError(
            f"Final {col} missing count changed."
        )


# HUD080 separately

s = csv_df["HUD080"]


if not (
    int(
        s.between(
            1,
            5
        ).sum()
    ) == 1025

    and

    int(
        s.eq(6).sum()
    ) == 13

    and

    int(
        s.eq(77777).sum()
    ) == 0

    and

    int(
        s.eq(99999).sum()
    ) == 2

    and

    int(
        s.isna().sum()
    ) == 9109
):

    raise ValueError(
        "Final HUD080 distribution changed."
    )


print(
    "\nPASS: Final HUQ_E frequencies "
    "match original XPT/CDC."
)


# ============================================================
# 33. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HUQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "HUQ050 = 1,449"
)

print(
    "TOTAL  = 1,449"
)


print(
    "\nGenuine decimal variables:",
    0
)


print(
    "\nCDC CHECK ITEMS not added:"
)

print(
    "HUQ015"
)

print(
    "HUQ085"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "HUQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 HUQ_E XPT -> CSV
Rows: 10,149
Columns: 10
PASS: HUQ_E structure = 10,149 rows x 10 columns.
PASS: HUQ015 and HUQ085 correctly absent from released XPT.

Integer/code/count fields: 10
Genuine decimal fields: []

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 1,449

Tiny values by variable:
HUQ050    1449

PASS: Exactly 1,449 tiny artifacts found, all in HUQ050.
PASS: CDC confirms HUQ050 permits 0 = None.

PASS: HUQ050 1,449 tiny values -> 0.
Remaining tiny artifacts: 0
PASS: No tiny artifacts remain.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

--- CDC FREQUENCY CHECK ---
PASS: HUQ010
PASS: HUQ020
PASS: HUQ030
PASS: HUQ040
PASS: HUQ050
PASS: HUQ060
PASS: HUQ071
PASS: HUQ090

PASS: HUD080 matches CDC.
      1-5 = 1,025
      6 or more = 13
      99999 = 2
      Missing = 9,109

PASS: HUD080 public-release codi